In [5]:
%load_ext autoreload
%autoreload 2

#global imports
from tensorboard.backend.event_processing import event_accumulator
import os
import scipy
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import nn, optim
from torch.nn import functional as F
from torch.utils.tensorboard import SummaryWriter
import matplotlib.pyplot as plt
import copy
from tqdm import tqdm
import csv
from scipy.special import softmax
import random
import logging
logger = logging.getLogger(__name__)
logging.basicConfig(filename='train_log.log', level=logging.INFO)
import itertools
import pandas as pd
import pickle
from sklearn import preprocessing
import pytorch_warmup as warmup

#local imports
from main.Dataset import ImportanceDataset, RealImportanceDataset, RealPredictionDataset, XBoxDatasetSimulation, Axios_ipsosdataset, HouseholdPulse_dataset
from main.GAN import GAN, WGAN_GP
from main.Discriminator import DataDiscriminator, DeepSetCritic
from main.util import set_seed
from DataProcessing import *
from main.experiments import *

#variable setups
device = torch.device('cuda:0')

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
SETTING SEED:  [134242]


In [ ]:
%autoreload 2
#Train RandomGAN

#STEP 1 storage path setups
path_to_census_dataset = ""
path_to_survey_dataset = ""
save_path = ""
log_path = ""

#STEP 2 RUN
train(census_dataset_path='./data/censusHouseholdPulse_data/cleaned/ipums_cleaned.csv',
      survey_dataset_path='./data/censusHouseholdPulse_data/cleaned/pulse_week'+str(29)+'_cleaned.csv',
      save_path="./saves/",
      save_dataset=True,
      save_weights=True,
      num_trials = 2)

STARTING TRAINING OF 2  TOTAL RUNS
SETTING SEED:  [114373]
-----------------------------------
RUN: 0 / 2

   RECVDVACC  REGION  EDUC  INCTOT  SEX  MARST  RACE  AGE
0          0       2     5       7    1      1     1    4
1          1       2     2       3    1      2     1    4
2          1       2     5       4    1      5     3    1
3          1       2     5       3    1      3     2    4
4          1       4     3       1    2      3     1    3
uniform avg target:  1.0
in warmup
in warmup


100%|██████████| 10/10 [00:01<00:00,  7.30it/s]


-----------------------------------
RUN: 1 / 2

   RECVDVACC  REGION  EDUC  INCTOT  SEX  MARST  RACE  AGE
0          0       2     5       7    1      1     1    4
1          1       2     2       3    1      2     1    4
2          1       2     5       4    1      5     3    1
3          1       2     5       3    1      3     2    4
4          1       4     3       1    2      3     1    3
uniform avg target:  0.0
in warmup
in warmup


100%|██████████| 10/10 [00:01<00:00,  7.20it/s]


In [ ]:
#entropy vs prediction power visualization
from scipy.ndimage import gaussian_filter1d
from scipy.signal import find_peaks
from itertools import combinations

#analyzing consistency over different types of randomness
'''
runs_consistency_baseline_seed:359556
runs_consistency_diffDataSeen_seed:359556
runs_consistency_diffNetworkInit_seed:359556
'''
#plotting events from saved run logs
from tensorboard.backend.event_processing import event_accumulator
import os
# Path to the directory where SummaryWriter saved logs

def plot_2d_runs(data):
    # Compute mean and standard deviation across experiments
    mean = np.mean(data, axis=0)
    std = np.std(data, axis=0)  # Or use std / sqrt(n) for standard error

    # Time steps (x-axis)
    time_steps = np.arange(data.shape[1])

    # Plot
    plt.figure(figsize=(10, 5))
    plt.plot(time_steps, mean, label='Mean Time Series')
    plt.fill_between(time_steps, mean - std, mean + std, alpha=0.3, label='±1 Std Dev')
    plt.xlabel('Time Step')
    plt.ylabel('Value')
    plt.title('Average Time Series with Error Bands')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

def load_runs_as_numpy(runs_path, var_names, filter_by, num_runs=9):
    run_folders = []
    count = 0
    for name in os.listdir(runs_path):
        valid_file = True
        for fb in filter_by:
            if fb not in name:
                valid_file = False
                break
        if not valid_file:
            continue
        if os.path.isdir(os.path.join(runs_path,name)):
            run_folders.append(os.path.join(runs_path,name))
            count += 1
        if count >= num_runs:
            break

    event_files = []
    for run_folder in run_folders:
        for name in os.listdir(run_folder):
            if os.path.isdir(os.path.join(run_folder,name)):
                continue
            else:
                event_files.append(os.path.join(run_folder,name))

    all_data = {}
    for i_name, vname in enumerate(var_names):
        all_data[vname] = []
        for i, event_file in enumerate(event_files):
            # Load event accumulator
            ea = event_accumulator.EventAccumulator(event_file)
            ea.Reload()

            # List available tags (scalars, histograms, images, etc.)
            # Read scalar values (e.g., 'loss', 'accuracy')
            prediction_events = ea.Scalars(vname)
            #summ = 0
            #for iter, event in enumerate(prediction_events):
            #    print(event, l2_events[iter])
            cur_pred = []
            for event in prediction_events:
                cur_pred.append(event.value)
            all_data[vname].append(cur_pred)
        all_data[vname] = np.array(all_data[vname])
    return all_data

def load_all_as_dict(runs_path, var_names, filter_by):
    run_folders = []
    count = 0
    all_names = []
    for name in os.listdir(runs_path):
        if os.path.isdir(os.path.join(runs_path,name)):
            run_folders.append(os.path.join(runs_path,name))
            all_names.append(name)
            count += 1

    event_files = []
    for run_folder in run_folders:
        for name in os.listdir(run_folder):
            if os.path.isdir(os.path.join(run_folder,name)):
                continue
            else:
                event_files.append(os.path.join(run_folder,name))

    all_data = {}
    for i_name, vname in enumerate(var_names):
        all_data[vname] = []
        for i, event_file in enumerate(event_files):
            # Load event accumulator
            ea = event_accumulator.EventAccumulator(event_file)
            ea.Reload()

            # List available tags (scalars, histograms, images, etc.)
            # Read scalar values (e.g., 'loss', 'accuracy')
            prediction_events = ea.Scalars(vname)
            #summ = 0
            #for iter, event in enumerate(prediction_events):
            #    print(event, l2_events[iter])
            cur_pred = []
            for event in prediction_events:
                cur_pred.append(event.value)
            all_data[vname].append(cur_pred)
        all_data[vname] = np.array(all_data[vname])
    return all_data, all_names

def print_stats(runs_path):
    all_data = load_runs_as_numpy(runs_path, ['Vaccine prediction', 'l2 norm demo diff'])
    all_vac_predictions = all_data['Vaccine prediction']
    all_l2_demographics = all_data['l2 norm demo diff']
    vac_stds = np.std(all_vac_predictions,axis=0)
    l2_stds = np.std(all_l2_demographics,axis=0)
    print("Consistency of Vac | consistency of L2 (30 trials)")
    print(np.mean(vac_stds),np.mean(l2_stds))
    print("")

    print("Lowest point of vacc variance")
    print(np.argmin(vac_stds[25:]), np.min(vac_stds[25:]), np.mean(all_vac_predictions[:,np.argmin(vac_stds[25:])]))

    #plot_2d_runs(all_vac_predictions[:,25:])

def find_peaks_troughs(runs_path, filter_by):
    sigma = 25  # smoothing level
    mr = 30
    out = load_runs_as_numpy(runs_path,  
                             ['Vaccine prediction total', 'GenLoss', 'Gen Entropy'],
                             filter_by=filter_by)
    vpt = out['Vaccine prediction total']
    gloss = out['GenLoss']
    entropy = out['Gen Entropy']
    for i, y in enumerate(gloss):
        # Smooth the signal
        smoothed = gaussian_filter1d(y, sigma=sigma)
            # Find peaks and troughs
        peaks, _ = find_peaks(smoothed)
        troughs, _ = find_peaks(-smoothed)

        converted_peaks = [int(p/2) for p in peaks]
        converted_troughs = [int(t/2) for t in troughs]
        #print(vpt.shape, converted_peaks, converted_troughs)
        peak_means = [np.mean(vpt[i,max(cp-mr,0):min(cp+mr,vpt.shape[1])]) for cp in converted_peaks]
        trough_means = [np.mean(vpt[i,max(ct-mr,0):min(ct+mr,vpt.shape[1])]) for ct in converted_troughs]

        print(np.mean(vpt[i, :]), trough_means)

def test(runs_path, filter_by):
    sigma = 25  # smoothing level
    mr = 30
    out = load_runs_as_numpy(runs_path,  
                             ['Vaccine prediction total', 'GenLoss', 'Gen Entropy'],
                             filter_by=filter_by)
    vpt = out['Vaccine prediction total']
    gloss = out['GenLoss']
    entropy = out['Gen Entropy']

    gloss_norms = np.linalg.norm(gloss, axis=1, keepdims=True)
    normalized_gloss = gloss / gloss_norms

    entropy_norms = np.linalg.norm(entropy, axis=1, keepdims=True)
    normalized_entropy = entropy / entropy_norms
    summed_arr = np.zeros(vpt.shape)
    for i, y in enumerate(normalized_entropy):
        for j in range(len(y)):
            summed_arr[i,j] = y[j] #normalized_gloss[i, j*2] + y[j]

        smoothed = gaussian_filter1d(summed_arr[i,:], sigma=sigma)
        mins = [np.argmin(smoothed)]

        trough_means = [np.mean(vpt[i,max(ct-mr,0):min(ct+mr,vpt.shape[1])]) for ct in mins]
        print(trough_means)
        print("")

    #idea 1 add gloss to entropy and find the point at which they are the lowest

def entropy_pred_relationship(runs_path, filter_by, num_runs):
    out = load_runs_as_numpy(runs_path,  
                             ['Vaccine prediction total', 'GenLoss', 'Gen Entropy', 'Truth - Fake scores', 'tvd'],
                             filter_by=filter_by,
                             num_runs=num_runs)
    vpt = out['Vaccine prediction total']
    gloss = out['GenLoss']
    entropy = out['Gen Entropy']
    t_f_scores = out['Truth - Fake scores']
    tvd = out['tvd']
    fig, axs = plt.subplots(3, 3, figsize=(10, 10))
    order = np.arange(vpt.shape[0])
    start_point = 0
    end_point = len(entropy[0])
    indices = np.arange(len(entropy[0,start_point:end_point]))

    sorted_vpt = np.array([
        x[np.argsort(y)] for x, y in zip(vpt, entropy)
    ])
    row_means = sorted_vpt[:, :10].mean(axis=1)

    # Mean of those means
    final_mean = row_means.mean()
    print(final_mean)
    return

    labels = ['entropy', 'truth-fake', 'tvd']
    for i in order[0:9]:
        if i > vpt.shape[0]:
            break
        row, col = divmod(i, 3)
        sc = axs[row,col].scatter(entropy[i,start_point:end_point],vpt[i,start_point:end_point],c=indices, cmap='viridis')
        #sc = axs[row+1,col].scatter(t_f_scores[i,start_point:end_point],vpt[i,start_point:end_point],c=indices, cmap='viridis')
        #sc = axs[row+2,col].scatter(tvd[i,start_point:end_point],vpt[i,start_point:end_point],c=indices, cmap='viridis')
        #fig.text(0.05, 0.85 - i * 0.3, labels[i], va='center', ha='right', fontsize=12)

    if sc is not None:
        cbar = fig.colorbar(sc, ax=axs, orientation='vertical', shrink=0.8)
        cbar.set_label('Time Index')

def success_rate_by_window_comparison(runs_path, filter_by, num_runs, window_length):
    """
    Compares the average of the first K and last K points for each row in a 2D array.

    Parameters:
    - loss_array: np.ndarray of shape (num_runs, time_steps)
    - K: int, number of points to average at the start and end

    Returns:
    - success_percent: float, percentage of rows where avg(first K) > avg(last K)
    """
    out = load_runs_as_numpy(runs_path,  
                             ['tvd', 'GenLoss', 'Gen Entropy'],
                             filter_by=filter_by,
                             num_runs=num_runs)
    loss_array = out['tvd']
    if window_length <= 0 or window_length > loss_array.shape[1] // 2:
        raise ValueError("K must be > 0 and <= half the number of time steps per row")

    start_avg = np.mean(loss_array[:, :window_length], axis=1)
    end_avg = np.mean(loss_array[:, -window_length:], axis=1)
    success_mask = start_avg > end_avg
    success_percent = 100 * np.mean(success_mask)

    return success_percent

def success_rate_JSD_loss(runs_path, filter_by, num_runs, window_length,override_loss_array=None):
    if override_loss_array is None:
        out = load_runs_as_numpy(runs_path,  
                                ['tvd', 'GenLoss', 'Gen Entropy'],
                                filter_by=filter_by,
                                num_runs=num_runs)
        loss_array = out['tvd']
    else:
        loss_array = override_loss_array
    nearly_identical = return_success_proportions(loss_array, 
                        0.03,
                        window_length)
    excellent = return_success_proportions(loss_array, 
                        0.05,
                        window_length)
    very_similar = return_success_proportions(loss_array, 
                        0.07,
                        window_length)
    print("Nearly Identical (0.03): ", nearly_identical)
    print("")
    print("Excellent (0.05): ", excellent)
    print("")
    print("Very similar (0.07): ", very_similar)
    return loss_array

def return_success_proportions(loss_array, 
                               threshold,
                               window_length):
    end_avg = np.mean(loss_array[:, -window_length:], axis=1)
    success_mask = end_avg <= threshold
    success_percent = 100 * np.mean(success_mask)
    mask = np.any(loss_array < threshold, axis=1)
    # Calculate percentage
    percent = 100 * np.sum(mask) / loss_array.shape[0]
    return success_percent, percent

def create_joint_table(runs_path, 
                       threshold,
                       filter_by,
                       var_list):
    out, names = load_all_as_dict(runs_path,  
                                ['tvd', 'GenLoss', 'Gen Entropy'],
                                filter_by=filter_by,)
    loss_array = out['tvd']

    results_dict = {}
    for name, loss in zip(names, loss_array):
        #determine what vars are in the name
        print(loss[0])
        var_count = 0
        c_vars = []
        for v in var_list:
            if v in name:
                var_count += 1
                c_vars.append(v)
        if len(c_vars) > 2:
            print(c_vars)
            print("ERROR: should only be 2 variables")
            assert 1 == 0
        c_vars = sorted(c_vars)

        #create results key
        new_key = tuple(c_vars)
        new_result = int(np.any(loss < threshold, axis=0))

        #store result in list
        if new_key not in results_dict:
            results_dict[new_key] = []
        results_dict[new_key].append(new_result)
    data = results_dict
    unique_strings = sorted(set([s for pair in data.keys() for s in pair]))

    # Create an empty DataFrame
    table = pd.DataFrame(index=unique_strings, columns=unique_strings, dtype=float)

    # Fill in the DataFrame with averages
    for (x, y), values in data.items():
        avg = np.mean(values)
        table.loc[x, y] = avg

    print(table)
    return results_dict, table


#file_paths = ["./runs_allweeks/"]
file_paths = ["./runs/"] 

for runs_path in file_paths:
    for i in range(23,30):
        entropy_pred_relationship(runs_path,
                                filter_by=["Week="+str(i)], 
                                    num_runs=20,)

    '''
    success_rate = success_rate_by_window_comparison(runs_path,
                                                    filter_by="", 
                                                     num_runs=20,
                                                     window_length=30)
    print(success_rate)
    var_names = ['AGE', 'SEX', 'REGION', 'EDUC', 'MARST', 'RACE', 'INCTOT']
    pair_list = list(combinations(var_names, 2))
    for var in pair_list:
        print(str(var) + " results: ")
        jsd_succ = success_rate_JSD_loss(runs_path, 
                              filter_by=list(var),
                              num_runs=300,
                              window_length = 10)
        print("")
    
    var_names = ['AGE', 'SEX', 'REGION', 'EDUC', 'MARST', 'RACE', 'INCTOT']
    results_dict, table = create_joint_table(runs_path, 
                                            0.07,
                                            filter_by = "",
                                            var_list = var_names)
    '''

In [ ]:
#making new household census data

from HouseholdCensusDataProcessing import * 
census_df = None
survey_df = None

vacc = {1:1,
        2:0}

for w in range(23,30):
    #print("processing: ", w)
    week = str(w)
    #census_df = pd.read_csv("./data/censusHouseholdPulse_data/usa_00008.csv")
    survey_df = pd.read_csv("./data/censusHouseholdPulse_data/pulse2021_puf_"+week+".csv")
    #survey_df, census_df = recoding_survey_and_census_data(survey_df, census_df)
    survey_df['RECVDVACC']=survey_df['RECVDVACC'].map(vacc)
    #survey_df.to_csv('./data/censusHouseholdPulse_data/pulse_week'+week+'_cleaned.csv', index=False)
    #census_df.to_csv('./data/censusHouseholdPulse_data/ipums_cleaned.csv',index=False)
    weighted_avg = (survey_df['RECVDVACC'] * survey_df['PWEIGHT']).sum() / survey_df['PWEIGHT'].sum()
    print(week, weighted_avg)